# Data Collection and Classification Challenge - YOLOv8

You have been asked to develop a deep-learning model that classifies UK aerial roof images as **flat** or **pitched**.

Collect additional images and fine-tune the pretrained `yolov8n-cls.pt` classification model. You may choose your training hyperparameters and use suitable data augmentation, but your experiments must be recorded as controlled comparisons.

Your best model checkpoint will be **evaluated on a hidden test set** — make sure it generalises beyond your local validation data.

## Data collection instructions

Use the supplied aerial-roof dataset and retain its existing filenames and folder structure:

```text
aerial/
|-- train/
|   |-- flat/
|   `-- pitched/
|-- val/
|   |-- flat/
|   `-- pitched/
`-- test/
    |-- flat/
    `-- pitched/
```

You may collect images from Google Earth, OpenAerialMap, Mapillary, Kaggle, public GIS/satellite portals, or your own permitted imagery.

Requirements:

* add **no more than 200 newly collected original images** in total;
* use clear, correctly cropped roof images and maintain a reasonable class balance;
* augmented copies, neighbouring crops, and repeated views of the same building do not count as separate original images;
* do not duplicate an image across `train`, `val`, and `test`;
* keep the filenames of all supplied images unchanged;
* use only sources that permit educational use.

Label each image as `flat` or `pitched` and place it in the appropriate folder. The main collection workflow and folder structure are otherwise unchanged from the original exercise.


In [ ]:
# Install Ultralytics into the active Jupyter kernel.
%pip install -q ultralytics

from ultralytics import YOLO
from pathlib import Path
from PIL import Image
from torchvision import transforms

import json
import random
import re
import shutil
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

try:
    from google.colab import files
except ImportError:
    files = None

CID = re.sub(r"[^0-9]", "", input("Enter your College ID: ").strip())
if not CID:
    raise ValueError("Use digits only for the College ID.")

DATA_ROOT = Path("/content/aerial")
SUBMISSION_ROOT = Path("/content/exercise10_submission")
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = SUBMISSION_ROOT
TABLE_DIR = SUBMISSION_ROOT
MODEL_DIR = SUBMISSION_ROOT
for directory in (FIGURE_DIR, TABLE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASSES = ("flat", "pitched")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
SEED = 42

# In Colab, upload aerial.zip and extract it before continuing:
# uploaded = files.upload()
# !unzip -q -o aerial.zip -d /content/


In [ ]:
def list_images(root, split, label):
    folder = root / split / label
    assert folder.exists(), f"Missing folder: {folder}"
    return sorted(
        path for path in folder.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )


count_rows = []
all_training_images = []
for split in ("train", "val", "test"):
    for label in CLASSES:
        paths = list_images(DATA_ROOT, split, label)
        count_rows.append({"Split": split, "Class": label, "Images": len(paths)})
        if split == "train":
            all_training_images.extend(paths)

dataset_counts = pd.DataFrame(count_rows)
display(dataset_counts)

random.Random(SEED).shuffle(all_training_images)
sample_paths = all_training_images[:6]
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for axis, path in zip(axes.flat, sample_paths):
    axis.imshow(Image.open(path).convert("RGB"))
    axis.set_title(path.parent.name)
    axis.axis("off")
fig.suptitle("Training-data examples")
fig.tight_layout()
plt.show()


## Introduction to data augmentation

Data augmentation creates modified training views without collecting new roofs. It can improve robustness to viewpoint, crop, lighting, and image-quality variation, but transformations must preserve the roof label.

Useful YOLO classification arguments include:

| Argument | Purpose | Example values |
|---|---|---|
| `auto_augment` | policy-based colour and geometric transformations | `None`, `"randaugment"`, `"ta_wide"` |
| `erasing` | remove a random image region | `0.0-0.5` |
| `mixup` | blend two images and labels | `0.0-0.2` |
| `cutmix` | paste a region from another image | `0.0-0.2` |
| `fliplr` | horizontal flip probability | `0.0-0.5` |
| `degrees` | random rotation range | `0-20` |
| `translate` | random translation fraction | `0.0-0.15` |
| `scale` | random image scaling | `0.0-0.3` |

Moderate transformations are appropriate. Avoid policies that obscure most of the roof, introduce unrealistic colours, or make the roof type ambiguous.

The following cell demonstrates augmentation visually. It is teaching content and does not add images to the submitted data collection.


In [ ]:
assert all_training_images, "Run the data-check cell first."
example_image = Image.open(all_training_images[0]).convert("RGB")

augmentation_demo = transforms.Compose([
    transforms.RandomResizedCrop(example_image.size, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
])

generator = torch.Generator().manual_seed(SEED)
torch.manual_seed(SEED)
views = [example_image] + [augmentation_demo(example_image) for _ in range(5)]

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for index, (axis, image) in enumerate(zip(axes.flat, views)):
    axis.imshow(image)
    axis.set_title("Original" if index == 0 else f"Augmented view {index}")
    axis.axis("off")
fig.suptitle("Examples of label-preserving data augmentation")
fig.tight_layout()
plt.show()


## Training and controlled parameter exploration

First run the supplied baseline configuration. Then complete at least **two controlled variations**.

For each variation:

* state a short hypothesis;
* change one primary setting, or one coherent augmentation policy;
* keep the dataset and all unrelated settings fixed;
* use validation accuracy to select the best checkpoint.

Possible comparisons include image size, batch size, optimiser, learning-rate schedule, and the augmentation arguments introduced above.

The final result table must contain:

* run ID and hypothesis;
* change relative to the baseline;
* image size, epochs, batch size, optimiser, and augmentation policy;
* best validation top-1 accuracy and best epoch;
* training time and selected checkpoint.


### Required evidence

Submit:

* `<CID>_E10_data.zip` — your **original** collected images only (do not include augmented copies) in the prescribed `train/val/test` structure;
* `<CID>_E10_best.pt` — your best model checkpoint (must use this exact filename), evaluated on a hidden test set;
* `<CID>_E10_experiment_results.csv` — the experiment results table (CSV) containing run ID, hypothesis, change from baseline, key parameters, best validation top-1 accuracy, best epoch, and training time;
* `<CID>_E10_validation_accuracy_vs_time.png` — a scatter plot of validation top-1 accuracy against training time;
* `<CID>_E10_final_learning_curves.png` — the best model's validation accuracy and loss curves over epochs.

The table and figures must be understandable without a separate report. Use a descriptive title, readable labels, a complete legend, and units where applicable.

Each submitted figure must include a **brief caption** (1–3 sentences) discussing the key findings — not just what is shown, but what the results mean. For example: which configuration achieves the best accuracy-to-time trade-off, and does data augmentation help?

Save the final files using the exact names above (replace `<CID>` with your College ID).

In [ ]:
BASELINE_CONFIG = {
    "model": "yolov8n-cls.pt",
    "imgsz": 224,
    "epochs": 20,
    "batch": 64,
    "optimizer": "SGD",
    "lr0": 0.01,
    "cos_lr": False,
    "auto_augment": None,
    "erasing": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "fliplr": 0.5,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.1,
    "patience": 5,
    "seed": SEED,
}


def run_experiment(specification):
    configuration = {**BASELINE_CONFIG, **specification["changes"]}
    model_name = configuration.pop("model")
    model = YOLO(model_name)
    start = time.perf_counter()
    model.train(
        data=str(DATA_ROOT),
        device=DEVICE,
        workers=2,
        project="/content/exercise10_runs",
        name=specification["run_id"],
        exist_ok=True,
        verbose=False,
        plots=True,
        **configuration,
    )
    training_time = time.perf_counter() - start
    save_dir = Path(model.trainer.save_dir)
    history_path = save_dir / "results.csv"
    history = pd.read_csv(history_path)
    accuracy_column = next(
        column for column in history.columns
        if "top1" in column.lower()
    )
    best_row = history.loc[history[accuracy_column].idxmax()]
    checkpoint = MODEL_DIR / f"{CID}_E10_{specification['run_id']}_best.pt"
    shutil.copy2(save_dir / "weights" / "best.pt", checkpoint)
    return {
        "run_id": specification["run_id"],
        "hypothesis": specification["hypothesis"],
        "change_from_baseline": json.dumps(specification["changes"], sort_keys=True),
        "imgsz": configuration["imgsz"],
        "epochs": configuration["epochs"],
        "batch": configuration["batch"],
        "optimizer": configuration["optimizer"],
        "augmentation": json.dumps({
            key: configuration[key]
            for key in (
                "auto_augment", "erasing", "mixup", "cutmix",
                "fliplr", "degrees", "translate", "scale",
            )
        }),
        "best_validation_top1": float(best_row[accuracy_column]),
        "best_epoch": int(best_row["epoch"]),
        "training_time_s": training_time,
        "checkpoint": str(checkpoint),
        "history_file": str(history_path),
    }


In [ ]:
EXPERIMENTS = [
    {
        "run_id": "baseline",
        "hypothesis": "Reference result using the supplied configuration.",
        "changes": {},
    },
    {
        "run_id": "variation_1",
        "hypothesis": "",
        "changes": {
        },
    },
    {
        "run_id": "variation_2",
        "hypothesis": "",
        "changes": {
        },
    },
]

assert EXPERIMENTS[1]["changes"], "Complete variation_1 before training."
assert EXPERIMENTS[2]["changes"], "Complete variation_2 before training."


In [ ]:
EXPERIMENT_RESULTS = [run_experiment(specification) for specification in EXPERIMENTS]
results_frame = pd.DataFrame(EXPERIMENT_RESULTS)
display(results_frame)

table_columns = [
    "run_id",
    "hypothesis",
    "change_from_baseline",
    "imgsz",
    "epochs",
    "batch",
    "optimizer",
    "augmentation",
    "best_validation_top1",
    "best_epoch",
    "training_time_s",
    "checkpoint",
]
table_frame = results_frame[table_columns].copy()
table_frame["best_validation_top1"] = table_frame["best_validation_top1"].map(lambda value: f"{value:.4f}")
table_frame["training_time_s"] = table_frame["training_time_s"].map(lambda value: f"{value:.1f}")

table_frame.to_csv(TABLE_DIR / f"{CID}_E10_experiment_results.csv", index=False)
print(f"Saved experiment results table to: {(TABLE_DIR / f'{CID}_E10_experiment_results.csv').resolve()}")

fig, axis = plt.subplots(figsize=(6, 4.5))
axis.scatter(results_frame["training_time_s"], results_frame["best_validation_top1"])
for _, row in results_frame.iterrows():
    axis.annotate(
        row["run_id"],
        (row["training_time_s"], row["best_validation_top1"]),
        xytext=(5, 5),
        textcoords="offset points",
    )
axis.set_xlabel("Training time (s)")
axis.set_ylabel("Best validation top-1")
axis.set_title("Validation accuracy versus training time")
axis.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"{CID}_E10_validation_accuracy_vs_time.png", dpi=200, bbox_inches="tight")
plt.close(fig)

best_result = results_frame.loc[results_frame["best_validation_top1"].idxmax()]
final_checkpoint = MODEL_DIR / f"{CID}_E10_best.pt"
shutil.copy2(best_result["checkpoint"], final_checkpoint)

best_history = pd.read_csv(best_result["history_file"])
accuracy_column = next(column for column in best_history.columns if "top1" in column.lower())
loss_columns = [column for column in best_history.columns if "loss" in column.lower()]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(best_history["epoch"], best_history[accuracy_column])
axes[0].set_title("Validation top-1 accuracy")
axes[0].set_xlabel("Epoch")
axes[0].grid(alpha=0.25)
for column in loss_columns:
    axes[1].plot(best_history["epoch"], best_history[column], label=column)
axes[1].set_title("Training history")
axes[1].set_xlabel("Epoch")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"{CID}_E10_final_learning_curves.png", dpi=200, bbox_inches="tight")
plt.close(fig)


## Final model check

After selecting the configuration using validation evidence, evaluate the selected checkpoint on the local test split once. Do not use the local test result to choose additional settings.


In [ ]:
final_model = YOLO(MODEL_DIR / f"{CID}_E10_best.pt")
test_metrics = final_model.val(
    data=str(DATA_ROOT),
    split="test",
    imgsz=int(best_result["imgsz"]),
    device=DEVICE,
    verbose=False,
)
print(f"Local test top-1 accuracy: {100 * test_metrics.top1:.2f}%")


In [ ]:
# Package original collected images into the submission zip.
data_zip_path = SUBMISSION_ROOT / f"{CID}_E10_data.zip"
if data_zip_path.exists():
    data_zip_path.unlink()
shutil.make_archive(
    str(data_zip_path.with_suffix("")),
    "zip",
    root_dir=str(DATA_ROOT.parent),
    base_dir=DATA_ROOT.name,
)
print(f"Saved {data_zip_path}")
if files:
    files.download(str(data_zip_path))


## Exercise 10 submission requirements

Submit the following files directly (no zip, no report, no notebook). Every file name must include your CID and the exercise number (`E10`).

| File | Description |
|------|-------------|
| `<CID>_E10_data.zip` | Your **original** collected images only (do not include augmented copies) in the prescribed `train/val/test` structure |
| `<CID>_E10_best.pt` | Your best model checkpoint (must use this exact filename) — evaluated on a hidden test set |
| `<CID>_E10_experiment_results.csv` | Experiment results table |
| `<CID>_E10_validation_accuracy_vs_time.png` | Validation accuracy vs training time scatter plot |
| `<CID>_E10_final_learning_curves.png` | Best model learning curves |

The experiment table must contain the baseline and at least two controlled variations.
